In [ ]:
!pip install promptukit

In [ ]:
import promptukit as pk
from promptukit.questions import validate_question
from promptukit.exams import create_exam

# 2) Load a question bank (path relative to the repository root). If you're
# running outside the repository (for example from an installed package),
# fall back to the packaged sample dataset that ships with `promptukit`.
import os
bank_path = 'content/question_banks/example_sections.json'
if os.path.exists(bank_path):
   data = pk.load(bank_path)
else:
   # load packaged sample included with the installed package
   data = pk.load_resource('question_banks/example_sections.json')

# 3) Inspect the file (section-based vs flat list)
if 'sections' in data:
   print('Sections:', [s.get('title') for s in data['sections']])
   print('First question:', data['sections'][0]['questions'][0])
   # Extract all questions from sections for PDF generation and validation
   all_questions = []
   for section in data['sections']:
       all_questions.extend(section.get('questions', []))
   # Create a new data structure suitable for build_exam_pdf and validation
   exam_data = {'questions': all_questions}
elif 'questions' in data:
   print('Total questions:', len(data['questions']))
   print('First question:', data['questions'][0])
   exam_data = data # Already in the correct format
else:
   print('Unexpected file shape:', type(data))
   exam_data = {} # Handle unexpected case gracefully


# 4) Validate programmatically
# Validate the flattened exam_data
validation_result = validate_question.validate(exam_data)
if isinstance(validation_result, tuple) and len(validation_result) == 2:
    errors, warnings = validation_result
elif isinstance(validation_result, tuple) and len(validation_result) == 1:
    # If it's a tuple of length 1, assume it's just the errors
    errors = validation_result[0]
    warnings = []
else:
    # If it's not a tuple, assume it's just the errors, and no warnings
    errors = validation_result
    warnings = []

if errors:
   print('Validation errors:', errors)
else:
   print('Bank valid — warnings:', warnings)

# 5) Generate a PDF exam from the same bank (we already have `data` loaded
# above as a dict, so pass it directly). Note: PDF generation requires
# the `reportlab` package: `pip install reportlab`.
# Pass the 'exam_data' which contains a flattened list of questions
if exam_data:
    create_exam.build_exam_pdf(exam_data, 'notebooks/output_exam.pdf')
else:
    print('Cannot build PDF: no valid exam data found.')